In [37]:
# ===== Robust Alpha Vantage loader + CHF risk functions =====
# Dependencies: requests, pandas, numpy, pathlib

import os, io, time, json, hashlib, pathlib
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
load_dotenv()

API_KEY = os.getenv("api_key")

# --- guard API key early ---
if not API_KEY or not isinstance(API_KEY, str):
    raise RuntimeError(
        "Alpha Vantage API key not found. Set env var 'api_key' (or adjust the code to your variable name)."
    )

CACHE_DIR = pathlib.Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

# ---- Schema requirements (minimum columns we rely on) ----
REQ = {
    "TIME_SERIES_DAILY": {"timestamp", "close"},
    "TIME_SERIES_DAILY_ADJUSTED": {"timestamp", "close"},  # adjusted_close is optional preference
    "FX_DAILY": {"timestamp", "close"},
}

# ----------------- helpers (kept in full) -----------------

def _cache_path(function: str, key: str) -> pathlib.Path:
    safe_key = key.replace("/", "_").replace(":", "_").replace(" ", "_")
    fname = f"{function}_{safe_key}.csv"
    return CACHE_DIR / fname

def _is_fresh(path: pathlib.Path, ttl_hours: int) -> bool:
    if not path.exists():
        return False
    age_hours = (time.time() - path.stat().st_mtime) / 3600
    # print(f"Cache file {path} age: {age_hours:.2f} hours (TTL {ttl_hours} hours)")
    return age_hours <= ttl_hours

def _looks_like_json(payload: bytes) -> bool:
    return payload.lstrip()[:1] in (b"{", b"[")

def _valid_columns(df: pd.DataFrame, required: set[str]) -> bool:
    cols = {c.strip().lower() for c in df.columns}
    return required.issubset(cols)

def fetch_csv_robust(url: str, function: str, key: str, ttl_hours: int = 24) -> pd.DataFrame:
    """
    Robust CSV fetch with:
      - on-disk cache (TTL),
      - JSON throttle/error detection (does NOT overwrite cache),
      - schema validation against REQ[function],
      - atomic write on success.
    Returns a DataFrame with lowercase column names.
    """
    path = _cache_path(function, key)

    # Serve fresh cache if valid
    if _is_fresh(path, ttl_hours):
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        if not _valid_columns(df, REQ[function]):
            raise RuntimeError(f"Cached file schema mismatch for {function}:{key}. Columns={list(df.columns)}")
        return df

    # Pull from network
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    raw = resp.content

    # Detect JSON throttle/error; do not poison cache
    if _looks_like_json(raw):
        # Try to show a concise message
        try:
            msg = json.loads(raw.decode("utf-8", errors="ignore"))
        except Exception:
            msg = {"body_head": raw[:200].decode("utf-8", errors="ignore")}
        raise RuntimeError(f"Alpha Vantage returned JSON (throttle/error) for {function}:{key} -> {str(msg)[:180]}")

    # Parse CSV and normalize
    df = pd.read_csv(io.BytesIO(raw))
    df.columns = [c.strip().lower() for c in df.columns]
    if not _valid_columns(df, REQ[function]):
        raise RuntimeError(f"Unexpected CSV schema for {function}:{key}. Got {list(df.columns)}.\n" f"CSV head:\n{df.head(3).to_string(index=False)}")

    # Atomic-ish write
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        f.write(raw)
    os.replace(tmp, path)
    return df

def _pick_close_column(df: pd.DataFrame) -> pd.Series:
    # Prefer adjusted_close when available (better for split/dividend-adjusted series).
    # Fallback to close otherwise. Endpoint (DAILY vs DAILY_ADJUSTED) is chosen upstream
    if "adjusted_close" in df.columns:
        return df["adjusted_close"]
    return df["close"]

# --------------- public API: data assembly ----------------

def build_returns_matrix_in_chf(
    holdings: list[dict],
    lookback_days: int = 252,
    use_adjusted: bool = True,
    ttl_hours: int = 24,
    no_fx: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Build CHF daily returns matrix for the provided holdings.

    holdings: list of dicts with keys:
      - name: row/column label in outputs
      - symbol: Alpha Vantage symbol (e.g., 'SWDA.LON', 'IBM')
      - ccy: base currency of the asset price series (GBP/USD/EUR/CHF)
      - gbx: bool; if True, divide close by 100.0 (LSE pence)
      - position: number of shares held (float)

    Returns:
      rets_df: DataFrame of CHF log returns, T x N
      prices_df: DataFrame of CHF closes, T x N
      weights: Series aligned to columns in rets_df
    """
    # Pre-fetch FX once per currency (excluding CHF)
    fx_map: dict[str, pd.Series] = {}
    needed_ccy = sorted({h["ccy"].upper() for h in holdings if h["ccy"].upper() != "CHF"})
    for ccy in needed_ccy:
        url_fx = (
            "https://www.alphavantage.co/query"
            f"?function=FX_DAILY&from_symbol={ccy}&to_symbol=CHF"
            f"&apikey={API_KEY}&outputsize=full&datatype=csv"
        )
        fx_df = fetch_csv_robust(url_fx, "FX_DAILY", f"{ccy}CHF", ttl_hours=ttl_hours)
        # fx_df["timestamp"] = pd.to_datetime(fx_df["timestamp"], utc=True)
        # fx_s = fx_df.sort_values("timestamp").set_index("timestamp")["close"].rename(f"{ccy}CHF")
        fx_df["timestamp"] = pd.to_datetime(fx_df["timestamp"], utc=True)
        fx_df = fx_df.sort_values("timestamp").set_index("timestamp")
        fx_s = fx_df.loc[:, "close"].rename(f"{ccy}CHF")  # explicit, Series with name
        fx_map[ccy] = fx_s

    # Build CHF close series per asset
    chf_close = {}
    for h in holdings:
        name  = h["name"]
        sym   = h["symbol"]
        ccy   = h["ccy"].upper()
        gbx   = bool(h.get("gbx", False))

        fn = "TIME_SERIES_DAILY_ADJUSTED" if use_adjusted else "TIME_SERIES_DAILY"
        url_px = (
            "https://www.alphavantage.co/query"
            f"?function={fn}&symbol={sym}"
            f"&apikey={API_KEY}&outputsize=full&datatype=csv"
        )
        px_df = fetch_csv_robust(url_px, fn, sym, ttl_hours=ttl_hours)
        px_df["timestamp"] = pd.to_datetime(px_df["timestamp"], utc=True)
        px_df = px_df.sort_values("timestamp").set_index("timestamp")

        close_local = _pick_close_column(px_df)
        if gbx:
            # LSE pence → GBP. (Does not change returns, only magnitude.)
            close_local = close_local / 100.0

        if (ccy == "CHF") or no_fx:
            close_chf = close_local.rename(name)
        else:
            fx = fx_map[ccy]
            df = pd.concat([close_local.rename("p"), fx.rename("x")], axis=1).dropna()
            close_chf = (df["p"] * df["x"]).rename(name)

        chf_close[name] = close_chf
        # print(f'chf_close["{name}"] tail:\n{chf_close[name].tail()}')
        # print the last value in 'chf_close[name]
    # print(f'Last value: {chf_close}')



    # Align on common dates and restrict to lookback window
    prices_df = pd.DataFrame(chf_close).dropna()
    prices_df = prices_df.tail(lookback_days)
    rets_df = np.log(prices_df / prices_df.shift(1)).dropna()

    if prices_df.shape[0] < (lookback_days * 0.9):
        raise ValueError(
            f"After alignment only {prices_df.shape[0]} rows remain "
            f"(expected {lookback_days}). Data source may not have full history."
    )
    if rets_df.isna().any().any():
        raise ValueError("NaNs remained in returns after shift/drop; check data alignment.")
    if (prices_df <= 0).any().any():
        raise ValueError("Non-positive prices encountered; check source data.")

    values = {}
    for h in holdings:
        name = h['name']
        last_price = chf_close[name].iloc[-1]
        values[name] = h['position'] * last_price
        # print(h['name'], h['position'], f'{last_price:.2f}', f'{values[name]:.2f}')

    # sum the values in values
    total_val = sum(values.values())
    print(f"Total portfolio value (CHF): {total_val:.2f}")


    # Weights (CHF)
    # total_val = sum(float(h["value_chf"]) for h in holdings)
    w = pd.Series({h["name"]: float(values[h["name"]]) / total_val for h in holdings})
    w = w.reindex(rets_df.columns).fillna(0.0)
    if not np.isclose(w.sum(), 1.0, atol=1e-6):
        raise ValueError(f"Weights must sum to 1. Got {w.sum():.6f}"
                         "check postions input in holdings.")

    return rets_df, prices_df, w

# --------------- public API: portfolio risk ----------------

def portfolio_risk(rets_df: pd.DataFrame, weights: pd.Series) -> dict:
    """
    Compute annualized vols, correlation, covariance, portfolio vol,
    marginal risk contribution (MRC), and percent risk contribution (PRC).
    """
    # Annualized stats
    cov_daily = rets_df.cov()
    cov_annual = cov_daily * 252.0
    vol_ann = rets_df.std() * np.sqrt(252.0)
    corr = rets_df.corr()

    # Align weights
    w = weights.reindex(rets_df.columns).astype(float)
    Sigma_w = cov_annual @ w
    port_var = float(w @ Sigma_w)
    port_vol = float(np.sqrt(port_var)) if port_var > 0 else 0.0

    # Contributions
    mrc = Sigma_w / port_vol if port_vol > 0 else Sigma_w * 0.0
    prc = (w * Sigma_w) / port_var if port_var > 0 else w * 0.0

    summary = pd.DataFrame({
        "Weight": w,
        "Vol_1Y_CHF": vol_ann,
        "MRC": mrc,           # ∂σ_p/∂w_i (absolute marginal contribution)
        "PRC_%": prc * 100.0  # percent contribution to total variance (sums ~100%)
    }).sort_values("PRC_%", ascending=False)

    return {
        "port_vol": port_vol,
        "cov_annual": cov_annual,
        "corr": corr,
        "vol_ann": vol_ann,
        "mrc": mrc,
        "prc": prc,
        "summary": summary,
    }

In [39]:

holdings = [
    {"name":"Unilever", "symbol":"ULVR.LON", "ccy":"GBP", "gbx":True, "position": 1266},
    {"name":"Shell",    "symbol":"SHEL.LON", "ccy":"GBP", "gbx":True, "position": 409 },
    {"name":"NatWest",  "symbol":"NWG.LON",  "ccy":"GBP", "gbx":True, "position": 377 },
    {"name":"Barclays", "symbol":"BARC.LON", "ccy":"GBP", "gbx":True, "position": 174 },
    {"name":"Tesco",    "symbol":"TSCO.LON", "ccy":"GBP", "gbx":True, "position": 4276},
    {"name":"SWDA",     "symbol":"SWDA.LON", "ccy":"GBP", "gbx":True, "position": 95  },
    {"name":"EMIM",     "symbol":"EMIM.LON", "ccy":"GBP", "gbx":True, "position": 199 },
    {"name":"IBM",      "symbol":"IBM",      "ccy":"USD", "gbx":False, "position": 4  },
    {"name":"ERNS",     "symbol":"ERNS.LON", "ccy":"GBP", "gbx":False, "position": 119},
]

TTL = 24
PERIOD = 252

rets_df, prices_df, w = build_returns_matrix_in_chf(holdings, lookback_days=PERIOD, use_adjusted=False, ttl_hours=TTL)

risk = portfolio_risk(rets_df, w)

# print(rets_df.tail())
print("Portfolio σ (annualized, CHF): {:.2%}".format(risk["port_vol"]))
print('MRC : Marginal Risk Contribution (CHF per 1% weight increase)')
print('PRC: Percent Risk Contribution (% of total portfolio variance)')
print(risk["summary"].round({"Weight":3,"Vol_1Y_CHF":3,"MRC":3,"PRC_%":1, }))
# Optional:
print(f'CORRELATION:')
print(f'{risk["corr"].round(2)}')
# print(f'COVARIANCE:')
# print(f'{risk["cov_annual"]}')
print('WITH FX IGNORED (in gbp)')
rets_df, prices_df, w = build_returns_matrix_in_chf(holdings, lookback_days=PERIOD, use_adjusted=False, ttl_hours=TTL, no_fx=True)
risk = portfolio_risk(rets_df, w)

print("Portfolio σ (annualized, GBP): {:.2%}".format(risk["port_vol"]))
print('MRC : Marginal Risk Contribution (gbp per 1% weight increase)')
print('PRC: Percent Risk Contribution (% of total portfolio variance)')
print(risk["summary"].round({"Weight":3,"Vol_1Y_CHF":3,"MRC":3,"PRC_%":1, }))
# Optional:
print(f'CORRELATION:')
print(f'{risk["corr"].round(2)}')

Total portfolio value (CHF): 128448.05
Portfolio σ (annualized, CHF): 13.10%
MRC : Marginal Risk Contribution (CHF per 1% weight increase)
PRC: Percent Risk Contribution (% of total portfolio variance)
          Weight  Vol_1Y_CHF    MRC  PRC_%
Unilever   0.498       0.183  0.163   61.8
Tesco      0.154       0.241  0.151   17.7
Shell      0.093       0.231  0.110    7.8
SWDA       0.073       0.171  0.072    4.0
ERNS       0.102       0.086  0.044    3.4
EMIM       0.051       0.172  0.071    2.8
NatWest    0.018       0.303  0.115    1.6
Barclays   0.006       0.352  0.133    0.6
IBM        0.006       0.311  0.075    0.3
CORRELATION:
          Unilever  Shell  NatWest  Barclays  Tesco  SWDA  EMIM   IBM  ERNS
Unilever      1.00   0.21     0.14      0.12   0.39  0.12  0.15  0.09  0.28
Shell         0.21   1.00     0.33      0.42   0.14  0.54  0.53  0.24  0.29
NatWest       0.14   0.33     1.00      0.83   0.19  0.57  0.54  0.16  0.36
Barclays      0.12   0.42     0.83      1.00   0.12